## Imports

In [1]:
%load_ext autoreload
%autoreload 2

# Standard imports
import glob

# 3rd party imports
import cv2
import matplotlib.pyplot as plt
import numpy as np 
from pprint import pprint
import SimpleITK as sitk
sitk.ProcessObject_SetGlobalWarningDisplay(False)
from scipy import ndimage

In [ ]:
# Shared helpers live in vessel_utils/ at the repository root.
# `pip install -e .` there makes the import below work anywhere; this is the fallback.
import pathlib, sys

_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "vessel_utils" / "__init__.py").exists()), None)
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from vessel_utils.io import load_channel
from vessel_utils.enhance import n4_bias_correction
from vessel_utils.viz import show


## Functions

In [3]:
### Functions for vessel detection
import itk
import numpy as np
from skimage.morphology import remove_small_objects, binary_closing, disk, remove_small_holes

In [4]:
### Functions for evaluation
import csv
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error
from scipy.spatial.distance import hamming

In [5]:
import time 
import scipy.ndimage
from scipy.ndimage import median_filter

In [ ]:
curr_img, fp = load_channel(filepath, IDX)
curr_ch1 = curr_img[:, :, 0]
curr_ch1 = curr_ch1.astype(np.float32)
plt.imshow(curr_ch1, cmap='gray')

## Load data

In [ ]:
import os
from skimage.measure import regionprops, label
from skimage.morphology import medial_axis, remove_small_holes
from pprint import pprint
import csv

# IO parameters
filepath = "/media/data/u01/Fig2025/Supplemental Fig3 DE/M14/*.tif"
output_csv_path = "/media/data/u01/Fig2025/quant-fig2025/Supple-3-DE/M14/"
output_image_path = "/media/data/u01/Fig2025/quant-fig2025/Supple-3-DE/M14/segmentation/"
IDX = 27  # 13, 14, 16 M1
N = 251
FILL_HOLES = True

# Load the image channels
curr_img, fp = load_channel(filepath, IDX)
curr_ch2 = curr_img[:, :, 0]
curr_ch2 = curr_ch2.astype(np.float32)
print("Filepath:", fp)

# Run N4 bias correction
bg_mask = np.ones(curr_ch2.shape, dtype=bool)
curr_ch2 = n4_bias_correction(curr_ch2, bg_mask, shrink_factor=2, show=False)

# Create a threshold mask for the image
curr_ch2_median = ndimage.median_filter(curr_ch2.copy(), size=3)  # Repeat for ch2
_, mask = cv2.threshold(curr_ch2_median.astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)
#mask2 = cv2.adaptiveThreshold(curr_ch2_median.astype(np.uint8), 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, N, 1)
#mask = mask * mask2
#mask[curr_ch2_median < thresh] = 0

mask = remove_small_objects(mask.astype(bool), min_size=200)
if FILL_HOLES:
     mask = remove_small_holes(mask, area_threshold=100)

# Compute the medial axis (skeleton) and the distance transform
skeleton_ch2, distance = medial_axis(mask, return_distance=True)

show(image=curr_ch2_median, title="CH2: Input image thresh",
     contour=mask,
     image2=curr_ch2_median, title2="CH2: Input image skeleton",
     contour2=skeleton_ch2,
     figsize=(20, 10),
     axis=False)


# Get the length of the skeletons
labeled_skeleton, num_features = label(skeleton_ch2, return_num=True)

# Save the skeleton info
thickness = []
lengths = []
for region in range(1, num_features + 1):
     region_mask = labeled_skeleton == region
     region_thickness = 2 * distance[region_mask]
     region_length = np.sum(region_mask)
     
     thickness.append(np.mean(region_thickness))
     lengths.append(region_length)


# Save segmentation to file
sitk_ch2 = sitk.GetImageFromArray(mask.astype(np.uint8))  # Ch2
output_ch2_file = output_image_path + os.path.basename(fp).replace(".tif", "_segmentation.tif")
sitk.WriteImage(sitk_ch2, output_ch2_file)

# Save skeleton to file
sitk_ch2 = sitk.GetImageFromArray(skeleton_ch2.astype(np.uint8))  # Ch2
output_ch2_file = output_image_path + os.path.basename(fp).replace(".tif", "_skeleton.tif")
sitk.WriteImage(sitk_ch2, output_ch2_file)

# Save skeleton lengths to csv
output_csv_file = output_csv_path + os.path.basename(fp).replace(".tif", ".csv")
with open(output_csv_file, mode='w') as csv_file:
    fieldnames = ['thickness', 'length']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    for i in range(len(thickness)):
        writer.writerow({'thickness': thickness[i], 'length': lengths[i]})